# Running ProSeD on your own model

This notebook builds a genotype-phenotype landscape from scratch, runs ProSeD
on it, and checks the result against the analytic theory. Nothing here depends
on the paper's data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from propgen import Landscape, equilibrium, simulate
from propgen.plotting import set_paper_style

set_paper_style(font_size=14)

## 1. Define a landscape

Three arrays: which genotypes mutate into which, how each genotype maps onto
phenotypes, and how fast each phenotype divides.

Note that genotype 1 expresses *both* phenotypes. That is the whole point:
under a classical model a genotype has one phenotype, and the dynamics below
would be different.

In [ ]:
landscape = Landscape(
    adjacency=np.array([[0.0, 1.0],
                        [1.0, 0.0]]),
    pheno_probs=np.array([[0.4, 0.6],      # genotype 0: 40% phenotype 0, 60% phenotype 1
                          [0.7, 0.3]]),    # genotype 1: 70% / 30%
    repro_probs=np.array([0.009, 0.002]),  # phenotype 0 divides ~4.5x faster
)
landscape

## 2. What does the theory predict?

In [ ]:
f_eq, mean_fitness = equilibrium(landscape, mutation_rate=0.1)

for g in range(2):
    for p in range(2):
        print(f"  f_eq[{g}, {p}] = {f_eq[g * 2 + p]:.4f}")
print(f"\n  mean fitness = {mean_fitness:.6g}")

## 3. Simulate

`n_cycles` counts serial dilutions, not generations: each cycle runs
`generations_per_cycle` rounds of reproduction before diluting back to
`pop_size`.

In [ ]:
result = simulate(
    landscape,
    n_cycles=400,
    pop_size=20_000,
    mutation_rate=0.1,
    generations_per_cycle=10,
    seed=0,
    progress=False,
)
result

## 4. Compare

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for g in range(2):
    for p in range(2):
        line, = ax.plot(result.cycles, result.frequencies[g, p],
                        lw=2, label=f"$g$={g}, $p$={p}")
        ax.axhline(f_eq[g * 2 + p], color=line.get_color(), ls="--", lw=2, alpha=0.5)

ax.set_xlabel("Dilution cycle")
ax.set_ylabel("Frequency")
ax.set_title("Solid: ProSeD simulation.   Dashed: analytic equilibrium.")
ax.legend()
plt.tight_layout()
plt.show()

observed = result.final_frequencies(last=100).ravel()
print(f"max |simulated - analytic| = {np.max(np.abs(observed - f_eq)):.5f}")

## 5. Change something

Make genotype 1 deterministic and rerun. The equilibrium moves, and the
low-fitness phenotype is no longer buoyed up by its genotype.

In [ ]:
deterministic = landscape.with_pheno_probs(1, [1.0, 0.0])
f_eq_det, _ = equilibrium(deterministic, mutation_rate=0.1)

print("probabilistic :", np.round(f_eq, 4))
print("deterministic :", np.round(f_eq_det, 4))

## Next steps

- Save a landscape to text files and drive it from a config: see `data/README.md`.
- Switch environments mid-run: see `configs/persister.yaml` and `propgen.schedule`.
- Run a parameter sweep in parallel: `propgen sweep --sweep ...`, see `experiments/`.